# Automation Bias, Algorithm Aversion, and Workflow Lab


## Purpose

This lab compares overreliance and underreliance under different workflow pressures.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1200

df = pd.DataFrame({
    "case_id": [f"HIAI-{i:04d}" for i in range(1, n + 1)],
    "model_confidence": rng.beta(5, 2, size=n),
    "explanation_quality": rng.beta(4, 3, size=n),
    "uncertainty_clarity": rng.beta(3.5, 3, size=n),
    "user_expertise": rng.choice(["novice", "intermediate", "expert"], size=n, p=[0.30, 0.45, 0.25]),
    "risk_level": rng.choice(["low", "medium", "high"], size=n, p=[0.45, 0.35, 0.20]),
    "time_pressure": rng.choice(["low", "medium", "high"], size=n, p=[0.35, 0.40, 0.25]),
})

df["model_correct"] = rng.binomial(1, np.clip(0.20 + 0.75 * df["model_confidence"], 0, 1), size=n)
df.head()

In [ ]:
pressure_boost = df["time_pressure"].map({"low": -0.05, "medium": 0.03, "high": 0.14})
risk_adjust = df["risk_level"].map({"low": 0.08, "medium": 0.00, "high": -0.10})

acceptance_probability = np.clip(
    0.12 + 0.50 * df["model_confidence"] + pressure_boost + risk_adjust,
    0.02,
    0.98,
)

df["accepted"] = rng.binomial(1, acceptance_probability, size=n)
df["overreliance"] = ((df["accepted"] == 1) & (df["model_correct"] == 0)).astype(int)
df["underreliance"] = ((df["accepted"] == 0) & (df["model_correct"] == 1)).astype(int)

df.groupby(["time_pressure", "risk_level"]).agg(
    cases=("case_id", "count"),
    acceptance_rate=("accepted", "mean"),
    overreliance_rate=("overreliance", "mean"),
    underreliance_rate=("underreliance", "mean"),
).reset_index()